# Lab 3 — Multi-Provider LLM APIs
**COM SCI-X 455 · Week 3 · Megan Armani, PhD**

> Push to `genai-course/week3/`

---

## Setup

In [8]:
!pip install openai anthropic rouge-score pydantic --quiet

import os
# Store keys in Colab Secrets (left sidebar lock icon) — never hardcode
from google.colab import userdata
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

## Part 1 — Basic API Calls: OpenAI + Anthropic + Llama 3

In [9]:
from openai import OpenAI
import anthropic

openai_client    = OpenAI()
anthropic_client = anthropic.Anthropic()

ARTICLE = """
The James Webb Space Telescope has captured the deepest infrared image of the universe ever taken.
The image shows thousands of galaxies, some dating back over 13 billion years.
Scientists believe this will revolutionise our understanding of early galaxy formation.
"""

In [3]:
# TODO: Call GPT-4o to summarise ARTICLE in 2 sentences.
# Use: openai_client.chat.completions.create(model='gpt-4o', messages=[...])
# Print the response.
gpt_resp = openai_client.chat.completions.create(
    model='gpt-4o',
    messages=[{"role": "user", "content": f"Summarise the following article in 2 sentences.\n{ARTICLE}"}],
)
gpt_summary = gpt_resp.choices[0].message.content
print(gpt_summary)

The James Webb Space Telescope has captured the deepest infrared image of the universe, revealing thousands of galaxies, some over 13 billion years old. This groundbreaking image is expected to revolutionize scientists' understanding of early galaxy formation.


In [12]:

from google.colab import userdata
from anthropic import Anthropic

anthropic_client = Anthropic(
    api_key=userdata.get("ANTHROPIC_API_KEY")
)

print("Anthropic client created successfully!")

Anthropic client created successfully!


In [14]:
# TODO: Call Claude (claude-sonnet-4-5) to summarise the same ARTICLE.
# Note: system= is a top-level param in Anthropic, NOT inside messages.
# Print and compare to GPT-4o output.
claude_resp = anthropic_client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=300,
    system="You are a precise summariser.",          # top-level param in Anthropic
    messages=[{"role": "user", "content": f"Summarise the following article in 2 sentences.\n{ARTICLE}"}],
)
claude_summary = claude_resp.content[0].text
print("Claude:\n" + claude_summary)
print("\nGPT-4o:\n" + gpt_summary)

Claude:
The James Webb Space Telescope has taken the deepest infrared image of the universe ever captured, revealing thousands of galaxies, some over 13 billion years old. Scientists expect this breakthrough to greatly advance our understanding of how early galaxies formed.

GPT-4o:
The James Webb Space Telescope has captured the deepest infrared image of the universe, revealing thousands of galaxies, some over 13 billion years old. This groundbreaking image is expected to revolutionize scientists' understanding of early galaxy formation.


## Part 2 — Streaming

In [18]:
# TODO: Implement streaming for GPT-4o: use stream=True in the API call.
# Print tokens as they arrive (print chunk, end='', flush=True).
# Measure: time to first token vs total time. Compare to non-streaming call.
import time

STREAM_PROMPT = "In about 150 words, explain why the James Webb Space Telescope matters for astronomy."
messages = [{"role": "user", "content": STREAM_PROMPT}]

# Streaming call
t0 = time.time()
first_token_time = None
stream = openai_client.chat.completions.create(model='gpt-4o', messages=messages, stream=True)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time() - t0
        print(chunk.choices[0].delta.content, end='', flush=True)
stream_total = time.time() - t0

# Non-streaming call
t0 = time.time()
openai_client.chat.completions.create(model='gpt-4o', messages=messages)
non_stream_total = time.time() - t0

print(f"\n\nStreaming:     time to first token = {first_token_time:.2f}s, total = {stream_total:.2f}s")
print(f"Non-streaming: user waits for the full response = {non_stream_total:.2f}s")

The James Webb Space Telescope (JWST) is a pivotal advancement for astronomy due to its ability to observe the universe in unprecedented detail. As the most powerful space telescope launched to date, JWST operates primarily in the infrared spectrum, allowing it to peer through cosmic dust clouds that obscure objects in visible light. This capability is crucial for studying the formation of stars and galaxies, including the earliest galaxies that formed over 13 billion years ago, shortly after the Big Bang. Moreover, JWST’s advanced instruments enable detailed spectroscopic analysis of exoplanet atmospheres, enhancing our understanding of their composition and potential habitability. Its innovative design, including a large segmented mirror and a sunshield to maintain low temperatures, ensures optimal performance and precision in detecting faint signals. By providing deeper insights into the universe's origins, structure, and potential for life beyond Earth, JWST significantly advances 

## Part 3 — Structured Output with Pydantic

In [19]:
from pydantic import BaseModel, field_validator
from typing import Literal
import json

class ArticleSummary(BaseModel):
    title:      str
    summary:    str   # exactly 2 sentences
    sentiment:  Literal['positive', 'negative', 'neutral']
    confidence: float # 0.0 to 1.0
    key_claims: list[str]  # max 3 items

    @field_validator('confidence')
    def confidence_range(cls, v):
        if not 0 <= v <= 1:
            raise ValueError('confidence must be 0-1')
        return v

    @field_validator('key_claims')
    def max_three(cls, v):
        if len(v) > 3:
            raise ValueError('max 3 key claims')
        return v

In [20]:
# TODO: Call GPT-4o with response_format={'type':'json_object'} to analyse ARTICLE.
# Ask it to return JSON matching ArticleSummary schema.
# Parse with ArticleSummary.model_validate_json(response).
# Show what happens when confidence=1.5 — does your validator catch it?
SCHEMA_PROMPT = f"""Analyse the article below and return ONLY a JSON object with these fields:
title (string), summary (exactly 2 sentences), sentiment ("positive", "negative" or "neutral"),
confidence (number between 0.0 and 1.0), key_claims (list of at most 3 strings).

Article:
{ARTICLE}"""

resp = openai_client.chat.completions.create(
    model='gpt-4o',
    response_format={'type': 'json_object'},
    messages=[{"role": "user", "content": SCHEMA_PROMPT}],
)
raw = resp.choices[0].message.content
result = ArticleSummary.model_validate_json(raw)
print(result)

# Failure case: confidence = 1.5
bad = json.loads(raw)
bad["confidence"] = 1.5
try:
    ArticleSummary.model_validate_json(json.dumps(bad))
except Exception as e:
    print("\nValidator caught confidence=1.5:\n", e)

title='James Webb Space Telescope Captures Deepest Infrared Image Ever' summary='The James Webb Space Telescope has captured the deepest infrared image of the universe ever taken, showing thousands of galaxies. Scientists believe this discovery will revolutionize our understanding of early galaxy formation.' sentiment='positive' confidence=0.9 key_claims=['The image shows thousands of galaxies, some dating back over 13 billion years.', 'Scientists believe this will revolutionise our understanding of early galaxy formation.']

Validator caught confidence=1.5:
 1 validation error for ArticleSummary
confidence
  Value error, confidence must be 0-1 [type=value_error, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


## Part 4 — Evaluation with ROUGE

In [21]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

REFERENCE = "The James Webb Space Telescope captured the universe's deepest infrared image, showing thousands of ancient galaxies that will transform our knowledge of early cosmic history."


In [22]:
# TODO: Score GPT-4o and Claude summaries against REFERENCE.
# Build a results table: Model | ROUGE-1 | ROUGE-2 | ROUGE-L

import pandas as pd

summaries = {
    "GPT-4o": gpt_summary,
    "Claude": claude_summary
}

rows = []

for name, text in summaries.items():
    s = scorer.score(REFERENCE, text)
    rows.append({
        "Model": name,
        "ROUGE-1": s['rouge1'].fmeasure,
        "ROUGE-2": s['rouge2'].fmeasure,
        "ROUGE-L": s['rougeL'].fmeasure
    })

rouge_table = pd.DataFrame(rows).round(3)

print(rouge_table.to_string(index=False))

print("\nHighest ROUGE-1:",
      rouge_table.loc[rouge_table["ROUGE-1"].idxmax(), "Model"])

 Model  ROUGE-1  ROUGE-2  ROUGE-L
GPT-4o    0.508    0.328    0.476
Claude    0.515    0.250    0.455

Highest ROUGE-1: Claude


### ❓ Reflect
1. Did GPT-4o, Claude, and Llama 3 produce meaningfully different summaries? Describe the differences.
2. Your ROUGE scores are based on one reference. What's wrong with this evaluation methodology?
3. What would a better evaluation framework look like for summarisation quality?

1. Did GPT-4o, Claude, and Llama 3 produce meaningfully different summaries?
I did not test Llama 3 because it kept giving me errors. GPT-4o and Claude produced very similar summaries, with minor differences in wording and phrasing.

2. What's wrong with using only one reference for ROUGE?
Using only one reference can be unfair because different summaries can express the same meaning using different words. ROUGE measures word overlap rather than actual meaning.

3. What would a better evaluation framework look like?
A better framework would use multiple reference summaries and evaluate factual accuracy and completeness. Human evaluation could also help determine which summary communicates the information best.

---
## Submission
- [ ] Push to `genai-course/week3/lab3.ipynb`
- [ ] All 3 models called and compared
- [ ] Pydantic validation demonstrated (including a failure case)
- [ ] ROUGE table completed

## Part 5 — Hallucination Detection
**Goal:** Identify hallucinated claims in model output. This feeds directly into your homework hallucination audit.

In [ ]:
# TODO: Design 3 prompts that are likely to trigger hallucination:
# 1. Ask for a specific statistic you know (e.g. 'What was OpenAI revenue in Q3 2019?')
# 2. Ask about a plausible-sounding but fictional event
# 3. Ask the model to cite a specific paper
#
# For each: run GPT-4o and Claude.
# Manually verify the claims. Mark each response:
#   CORRECT / HALLUCINATED / UNVERIFIABLE
# Build a table: Model | Prompt | Claim | Verdict
# What pattern do you see in when each model hallucinates?
import pandas as pd

HALLU_PROMPTS = [
    "What was the official attendance at Super Bowl I in 1967? Give the exact number.",
    "Describe the main provisions of the 2008 Helsinki Convention on Autonomous Vehicle Liability and list the countries that ratified it.",
    "Cite the specific paper (authors, year, venue, page numbers) that first showed layer normalization reduces transformer training time by 40%.",
]

def ask_gpt(p):
    r = openai_client.chat.completions.create(model='gpt-4o', temperature=0,
                                              messages=[{"role": "user", "content": p}])
    return r.choices[0].message.content

def ask_claude(p):
    r = anthropic_client.messages.create(model='claude-sonnet-4-5', max_tokens=500, temperature=0,
                                         messages=[{"role": "user", "content": p}])
    return r.content[0].text

# After reading the answers and checking them yourself, fill this in:
# (model, prompt number) -> "CORRECT" | "HALLUCINATED" | "UNVERIFIABLE"
VERDICTS = {}

if 'answers' not in globals():      # cache, so re-running after you fill VERDICTS does not call the APIs again
    answers = {}

rows = []
for i, p in enumerate(HALLU_PROMPTS, 1):
    for model, fn in [("GPT-4o", ask_gpt), ("Claude", ask_claude)]:
        if (model, i) not in answers:
            answers[(model, i)] = fn(p)
        answer = answers[(model, i)]
        print(f"\n=== {model} | Prompt {i} ===\n{answer}")
        rows.append({"Model": model, "Prompt": p[:60], "Claim": answer[:150].replace("\n", " "),
                     "Verdict": VERDICTS.get((model, i), "TODO")})
print()
print(pd.DataFrame(rows).to_string(index=False))

### ❓ Reflect — Hallucination
1. Which model hallucinated more? Was it confident when wrong?
2. What mitigation (RAG, temperature=0, structured output) would help most for your specific prompts?
3. 'The model made it up' is not an explanation. What IS the architectural reason hallucination occurs?

Reflection:

1. Which model hallucinated more? Was it confident when wrong?
Neither model clearly hallucinated. Both accurately summarized the article. However, GPT-4o used stronger wording like "revolutionize" while Claude used "greatly advance". Both models sounded confident but neither made an obvious factual error.

2. What mitigation would help most?
RAG would help most because it gives the models the original article to reference, reducing the chance of adding information that is not supported by the source.

3. What is the architectural reason hallucination occurs?
Language models predict the next most likely word based on patterns learned during training. They are designed to generate believable and coherent text, not automatically verify facts, that's why they can produce information that sounds correct but is factually wrong.